In [ ]:
from configs.datasets_config import get_dataset_info
from qm9.visualizer import load_molecule_xyz, plot_data3d, visualize_chain_uncertainty, load_molecule_xyz
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import qm9.visualizer as vis
from scipy.ndimage import gaussian_filter

dataset_info = get_dataset_info('qm9', False)

In [ ]:
# --- Input molecule ---
mol = {'num_atoms': torch.tensor(5), 'charges': torch.tensor([6, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0]), 'positions': torch.tensor([[-1.2698e-02,  1.0858e+00,  8.0010e-03],
        [ 2.1504e-03, -6.0313e-03,  1.9761e-03],
        [ 1.0117e+00,  1.4638e+00,  2.7657e-04],
        [-5.4082e-01,  1.4475e+00, -8.7664e-01],
        [-5.2381e-01,  1.4379e+00,  9.0640e-01],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00]]), 'one_hot': torch.tensor([[False,  True, False, False, False],
        [ True, False, False, False, False],
        [ True, False, False, False, False],
        [ True, False, False, False, False],
        [ True, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False]])}

one_hot = mol['one_hot'].float().unsqueeze(0)

num_atoms = mol['num_atoms']
charges = mol['charges'][:num_atoms].unsqueeze(0).unsqueeze(-1)

node_mask = torch.zeros(1, mol['charges'].shape[0])
node_mask[:, :num_atoms] = 1

x = mol['positions'].unsqueeze(0)

print("one_hot:", one_hot.shape)
print("charges:", charges.shape)
print("node_mask:", node_mask.shape)
print("positions:", x.shape)


In [ ]:
vis.save_xyz_file('', one_hot, charges, x,
        id_from=0, name='vis', dataset_info=dataset_info,
        node_mask=node_mask)

In [ ]:
mol = load_molecule_xyz('vis_000.txt', dataset_info)
mol

In [ ]:
import numpy as np

def forward_diffuse_xyz(input_path, schedule, t):
    """Apply forward diffusion to molecule coordinates in an XYZ file."""
    # --- Read xyz ---
    with open(input_path) as f:
        n = int(f.readline().strip())
        comment = f.readline().strip()
        elems, coords = [], []
        for _ in range(n):
            e, x, y, z = f.readline().split()
            elems.append(e)
            coords.append([float(x), float(y), float(z)])
    coords = np.array(coords)

    if schedule == "cosine":
        alpha = np.cos((t + 1e-5) / 1.00001 * np.pi / 2) ** 2
    elif schedule == "quadratic":
        alpha = (1 - t ** 2)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")

    # --- Add noise ---
    np.random.seed(42) 
    noise = np.random.normal(size=coords.shape)
    coords_t = np.sqrt(alpha) * coords + np.sqrt(1 - alpha) * noise

    # --- Write noisy xyz ---
    output_path = input_path.replace(".txt", f"_{schedule}_noised_t{t:.2f}.txt")
    with open(output_path, "w") as f:
        f.write(f"{n}\n{comment} (noised t={t})\n")
        for e, (x, y, z) in zip(elems, coords_t):
            f.write(f"{e} {x:.6f} {y:.6f} {z:.6f}\n")

    print(f"Saved: {output_path}  (t={t})")

In [ ]:
steps = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for t in steps:
    forward_diffuse_xyz('vis_000.txt', "cosine", t=t)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

def display_images_row(paths, figsize_per_image=(3, 3), interpolation='nearest'):
    """
    Display images in a horizontal row with no gaps.
    paths: list of file paths (strings)
    figsize_per_image: tuple (width, height) per image in inches
    """
    n = len(paths)
    if n == 0:
        raise ValueError("No image paths provided.")
    figsize = (figsize_per_image[0] * n, figsize_per_image[1])
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        img = Image.open(p).convert('RGB')   # works for PNG/JPG, converts grayscale to RGB
        ax.imshow(np.asarray(img), interpolation=interpolation)
        ax.axis('off')
    plt.subplots_adjust(wspace=0, hspace=0)  # remove gaps
    plt.show()

# Dummy usage:
file_paths = [
    "molecule_actual_noised_t1.00.jpg",
    "molecule_actual_noised_t0.80.jpg",
    "molecule_actual_noised_t0.70.jpg",
    "molecule_actual_noised_t0.50.jpg",
    "molecule_actual_noised_t0.40.jpg",
    "molecule_actual_noised_t0.20.jpg",
]

display_images_row(file_paths)


In [ ]:
# Load image
img = Image.open("dog.jpeg").resize((128, 128))
x0 = np.array(img).astype(np.float32) / 255.0

# Cosine schedule function
def cosine_schedule(t):
    return np.cos((t + 0.008) / 1.008 * np.pi / 2) ** 2

# Forward diffusion process
def forward_diffusion(x0, t, scale, noise=None):
    if noise is None:
        noise = np.random.randn(*x0.shape)
    alpha_t = cosine_schedule(t)
    return np.sqrt(alpha_t) * x0 * scale + np.sqrt(1 - alpha_t) * noise

# List of scale factors
scales = [0.8, 1.0, 1.2]
t = 0.1  # Fixed timestep

# Plotting
plt.figure(figsize=(9, 3))
for i, scale in enumerate(scales):
    xt = np.clip(forward_diffusion(x0, t, scale), 0, 1)
    plt.subplot(1, len(scales), i + 1)
    plt.imshow(xt)
    plt.axis('off')
    plt.title(f"scale={scale}, t={t}")

plt.subplots_adjust(wspace=0, hspace=0)
plt.show()
